# NB07b — Alternate proposed method (master pipeline)

**Method:** a single **BanglishBERT** encoder with the **full technique stack** —
CE → focal+CW → balanced sampler → MSD → R-Drop → FGM → EMA — **seed-ensembled** (Nelder-Mead on val).
BanglishBERT is bilingual, so there is **no script isolation** (it trains/tests on both scripts).

This notebook is **fully self-contained** — it trains fresh for every evaluation and never reads any
other notebook's logits (no leakage). It runs three legs and writes everything under `outputs/altmethod/`:

1. **Benchmark** — random 20% held-out test (directly comparable to the main CE+FGM ensemble).
2. **Robustness** — every `source_holdout_*` and `script_holdout_*` config, trained from scratch on that
   config's train+val and evaluated on its unseen test (hard `uid` leakage guard).
3. **Base-paper** — facebook-44k, native 5 classes, 70/15/15, head-to-head vs the base paper (89.23).

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True" 

In [ ]:
import os, re, json, glob, time, math, random, warnings, unicodedata
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score, classification_report
from scipy.optimize import minimize
warnings.filterwarnings("ignore")
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ── CONFIG — Alternate method: BanglishBERT + FULL technique stack ──────────
DEBUG=False; DEBUG_SAMPLES=400; DEBUG_EPOCHS=2
MODEL_NAME="csebuetnlp/banglishbert"
SEEDS=[42,123,456]                # seed-ensemble for benchmark + base-paper
ROBUSTNESS_SEEDS=[42]             # 6 robustness configs → 1 seed keeps it tractable; set =SEEDS for seed-ensemble
CFG={"max_length":128,"batch_size":32,"eval_batch_size":128,"grad_accum_steps":1,"num_workers":4,
     "epochs":8,"encoder_lr":2e-5,"head_lr":8e-5,"weight_decay":0.01,"warmup_ratio":0.10,
     "max_grad_norm":1.0,"label_smoothing":0.03,"focal_gamma":2.0,"class_weight_beta":0.999,
     "dropout":0.25,"head_hidden_dim":384,"n_msd":4,"patience":3,"use_fp16":torch.cuda.is_available(),
     # ── FULL stack ON ──
     "use_focal":True,"use_cw":True,"use_balanced_sampler":True,"sampler_alpha":0.5,
     "use_fgm":True,"fgm_epsilon":1.0,"use_rdrop":True,"rdrop_alpha":0.5,"use_ema":True,"ema_decay":0.999}
SPLIT_DIR="../data/splits"; RAW="../data/merged/benchmark_raw.csv"
OUT="../outputs/altmethod"; os.makedirs(OUT,exist_ok=True); SPLIT_SEED=42
BASE_PAPER={"macro_f1":0.8923,"per_class":{"Not Bully":0.9151,"Sexual":0.8845,"Troll":0.8446,"Religious":0.9374,"Threat":0.7579}}
if DEBUG: CFG["epochs"]=DEBUG_EPOCHS; SEEDS=[42]; ROBUSTNESS_SEEDS=[42]; print("⚠ DEBUG")
print(f"method: BanglishBERT full-stack | seeds={SEEDS} robustness_seeds={ROBUSTNESS_SEEDS}")
print(f"toggles: focal={CFG['use_focal']} cw={CFG['use_cw']} sampler={CFG['use_balanced_sampler']} "
      f"msd={CFG['n_msd']} rdrop={CFG['use_rdrop']} fgm={CFG['use_fgm']} ema={CFG['use_ema']} | batch={CFG['batch_size']}/accum{CFG['grad_accum_steps']}")

In [ ]:
# ── Machinery (validated; identical to NB05/07, full stack) ────────────────
class DS(Dataset):
    def __init__(self,df,tok,maxlen,enc,text_col,label_col):
        self.texts=df.reset_index(drop=True)[text_col].fillna("").astype(str).tolist()
        self.labels=[int(enc.get(v,-1)) for v in df[label_col]]; self.tok,self.maxlen=tok,maxlen
    def __len__(self): return len(self.texts)
    def __getitem__(self,i):
        e=self.tok(self.texts[i],max_length=self.maxlen,truncation=True,padding=False)
        it={k:torch.tensor(v,dtype=torch.long) for k,v in e.items()}; it["label"]=torch.tensor(self.labels[i],dtype=torch.long); return it
class Collator:
    def __init__(self,tok): self.tok=tok
    def __call__(self,fs):
        labels=torch.stack([f["label"] for f in fs]); feats=[{k:v for k,v in f.items() if k!="label"} for f in fs]
        b=self.tok.pad(feats,padding=True,return_tensors="pt"); b["label"]=labels; return b
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,ls=0.0): super().__init__(); self.g,self.w,self.ls=gamma,weight,ls
    def forward(self,lg,t):
        ce=F.cross_entropy(lg,t,weight=self.w,reduction="none",label_smoothing=self.ls); return (((1-torch.exp(-ce))**self.g)*ce).mean()
def build_cw(series,enc,beta=0.999,max_w=10.0):
    m=series.map(enc).dropna().astype(int); n=len(enc); c=m.value_counts().sort_index(); w=np.ones(n,dtype=np.float32)
    for i in range(n):
        k=int(c.get(i,0))
        if k>0: w[i]=1.0/max((1.0-beta**k)/max(1.0-beta,1e-12),1e-9)
    w=np.clip(w,w.min(),w.min()*max_w); w=w/w.mean(); return torch.tensor(w,dtype=torch.float32,device=device)
def make_sampler(df,enc,label_col,alpha=0.5):
    y=df[label_col].map(enc).fillna(-1).astype(int).values; cc=np.bincount(y[y>=0],minlength=len(enc)).astype(float); cc[cc==0]=1.0
    cw=(1.0/cc)**float(alpha); sw=np.where(y>=0,cw[np.clip(y,0,None)],0.0)
    return WeightedRandomSampler(torch.as_tensor(sw,dtype=torch.double),len(sw),replacement=True)
class MSDHead(nn.Module):
    def __init__(self,h,n,dropout=0.25,inner=384,n_msd=4):
        super().__init__(); inner=min(inner,h); self.pre=nn.Sequential(nn.Linear(h,inner),nn.GELU(),nn.LayerNorm(inner))
        self.drops=nn.ModuleList([nn.Dropout(dropout) for _ in range(max(1,n_msd))]); self.out=nn.Linear(inner,n)
    def forward(self,x):
        h=self.pre(x)
        if self.training and len(self.drops)>1: return torch.stack([self.out(d(h)) for d in self.drops],0).mean(0)
        return self.out(self.drops[0](h))
class Encoder(nn.Module):
    def __init__(self,name,n,dropout=0.25,inner=384,n_msd=4):
        super().__init__(); self.encoder=AutoModel.from_pretrained(name); h=self.encoder.config.hidden_size
        self._tti=self.encoder.config.model_type.lower() not in ("roberta","xlm-roberta"); self.head=MSDHead(h,n,dropout,inner,n_msd)
    def _pool(self,o,m):
        hs=o.last_hidden_state; cls=hs[:,0]; mean=(hs*m.unsqueeze(-1).float()).sum(1)/m.sum(1,keepdim=True).float().clamp(1); return 0.5*cls+0.5*mean
    def forward(self,ids,mask,tti=None):
        kw={"input_ids":ids,"attention_mask":mask}
        if self._tti and tti is not None: kw["token_type_ids"]=tti
        return self.head(self._pool(self.encoder(**kw),mask))
def uparams(model,e,h,wd):
    nd=["bias","LayerNorm.weight","LayerNorm.bias"]; g=[]
    for grp,lr in [(model.encoder,e),(model.head,h)]:
        d,nde=[],[]
        for n,p in grp.named_parameters():
            if p.requires_grad: (nde if any(x in n for x in nd) else d).append(p)
        g+=[{"params":d,"lr":lr,"weight_decay":wd},{"params":nde,"lr":lr,"weight_decay":0.0}]
    return g
class FGM:
    def __init__(self,model,eps=1.0,key="word_embeddings"): self.m,self.e,self.k,self.b=model,eps,key,{}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.k in n and p.grad is not None:
                self.b[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(self.e*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.b: p.data=self.b[n]
        self.b={}
class EMA:
    def __init__(self,model,decay=0.999): self.d=decay; self.s={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}; self.b={}
    def update(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.detach(),alpha=1-self.d)
    def apply_shadow(self,model):
        self.b={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
        for n,p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.s[n])
    def restore(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.b: p.data.copy_(self.b[n])
        self.b={}
def rdrop_kl(lp,lq):
    pl,ql=F.log_softmax(lp,-1),F.log_softmax(lq,-1); p,q=pl.exp(),ql.exp()
    return 0.5*((p*(pl-ql)).sum(-1)+(q*(ql-pl)).sum(-1)).mean()
@torch.no_grad()
def get_logits(model,loader):
    model.eval(); o=[]
    for b in loader:
        b={k:v.to(device) for k,v in b.items()}; o.append(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).cpu())
    return torch.cat(o)
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def metrics(yt,yp,pr,nc):
    rec={"macro_f1":round(float(f1_score(yt,yp,average="macro",zero_division=0)),4),
         "weighted_f1":round(float(f1_score(yt,yp,average="weighted",zero_division=0)),4),
         "accuracy":round(float(accuracy_score(yt,yp)),4),
         "mcc":round(float(matthews_corrcoef(yt,yp)),4)}
    try: rec["macro_auroc"]=round(float(roc_auc_score(yt,pr,multi_class="ovr",average="macro",labels=list(range(nc)))),4)
    except Exception: rec["macro_auroc"]=float("nan")
    return rec
print("machinery ok")

In [ ]:
# ── Train one BanglishBERT (full stack, val-guarded EMA) ───────────────────
def train_one(seed,train_df,val_df,enc,nc,save,text_col,label_col):
    set_seed(seed); os.makedirs(save,exist_ok=True); cfg=CFG
    tok=AutoTokenizer.from_pretrained(MODEL_NAME); coll=Collator(tok)
    lkw=dict(collate_fn=coll,num_workers=cfg["num_workers"],pin_memory=torch.cuda.is_available())
    ds=DS(train_df,tok,cfg["max_length"],enc,text_col,label_col)
    if cfg["use_balanced_sampler"]:
        tl=DataLoader(ds,batch_size=cfg["batch_size"],sampler=make_sampler(train_df,enc,label_col,cfg["sampler_alpha"]),shuffle=False,drop_last=True,**lkw)
    else:
        tl=DataLoader(ds,batch_size=cfg["batch_size"],shuffle=True,drop_last=True,**lkw)
    vl=DataLoader(DS(val_df,tok,cfg["max_length"],enc,text_col,label_col),batch_size=cfg["eval_batch_size"],shuffle=False,**lkw)
    model=Encoder(MODEL_NAME,nc,cfg["dropout"],cfg["head_hidden_dim"],cfg["n_msd"]).to(device)
    cw=build_cw(train_df[label_col],enc,cfg["class_weight_beta"]) if cfg["use_cw"] else None
    crit=FocalLoss(cfg["focal_gamma"],cw,cfg["label_smoothing"]) if cfg["use_focal"] else (lambda lg,t: F.cross_entropy(lg,t,weight=cw,label_smoothing=cfg["label_smoothing"]))
    opt=torch.optim.AdamW(uparams(model,cfg["encoder_lr"],cfg["head_lr"],cfg["weight_decay"]))
    ns=math.ceil(len(tl)/cfg["grad_accum_steps"])*cfg["epochs"]; sch=get_linear_schedule_with_warmup(opt,int(ns*cfg["warmup_ratio"]),ns)
    scaler=torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type=="cuda") else None
    fgm=FGM(model,cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema=EMA(model,cfg["ema_decay"]) if cfg["use_ema"] else None
    @torch.no_grad()
    def vmacro():
        model.eval(); P,Y=[],[]
        for b in vl:
            b={k:v.to(device) for k,v in b.items()}
            P.extend(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).argmax(-1).cpu().numpy()); Y.extend(b["label"].cpu().numpy())
        Y=np.array(Y); m=Y>=0; return f1_score(Y[m],np.array(P)[m],average="macro",zero_division=0)
    best,noimp,t0=-1.0,0,time.time()
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step,b in enumerate(tl,1):
            b={k:v.to(device) for k,v in b.items()}; y=b["label"]
            with torch.autocast(device_type=device.type,enabled=scaler is not None):
                l1=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    l2=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                    loss=0.5*(crit(l1,y)+crit(l2,y))+cfg["rdrop_alpha"]*rdrop_kl(l1,l2)
                else: loss=crit(l1,y)
                loss=loss/cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type,enabled=scaler is not None):
                    la=crit(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")),y)/cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step%cfg["grad_accum_steps"]==0 or step==len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["max_grad_norm"])
                (scaler.step(opt),scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        vm_raw=vmacro(); vm,use_ema=vm_raw,False
        if ema is not None and ep>=2:
            ema.apply_shadow(model); vm_ema=vmacro()
            if vm_ema>=vm_raw: vm,use_ema=vm_ema,True
            ema.restore(model)
        if vm>best:
            best,noimp=vm,0
            if use_ema: ema.apply_shadow(model)
            torch.save(model.state_dict(),f"{save}/best.pt")
            if use_ema: ema.restore(model)
        else: noimp+=1
        if noimp>=cfg["patience"]: break
    model.load_state_dict(torch.load(f"{save}/best.pt",map_location=device,weights_only=True))
    torch.save(get_logits(model,vl),f"{save}/val_logits.pt")
    print(f"    seed{seed}: best_val_macroF1={best:.4f} [{(time.time()-t0)/60:.1f}min]")
    return model,tok,best
print("train_one ok")

In [ ]:
# ── Run one leg: train BanglishBERT × seeds, seed-ensemble on test ─────────
def run_leg(train_df,val_df,test_df,enc,nc,tag,seeds,text_col="text_clean",label_col="label5"):
    save_root=f"{OUT}/{tag}"; os.makedirs(save_root,exist_ok=True)
    y_val=val_df[label_col].map(enc).values; y_test=test_df[label_col].map(enc).values
    val_lg,test_lg,singles={},{},[]
    for sd in seeds:
        save=f"{save_root}/seed{sd}"
        model,tok,bv=train_one(sd,train_df,val_df,enc,nc,save,text_col,label_col)
        coll=Collator(tok); lkw=dict(collate_fn=coll,num_workers=0)
        tel=DataLoader(DS(test_df,tok,CFG["max_length"],enc,text_col,label_col),batch_size=CFG["eval_batch_size"],shuffle=False,**lkw)
        val_lg[f"s{sd}"]=torch.load(f"{save}/val_logits.pt",map_location="cpu",weights_only=False).float()
        test_lg[f"s{sd}"]=get_logits(model,tel).float()
        sp=F.softmax(test_lg[f"s{sd}"],-1).numpy(); sy=test_lg[f"s{sd}"].argmax(-1).numpy()
        singles.append(metrics(y_test,sy,sp,nc)["macro_f1"])
        del model; torch.cuda.empty_cache()
    names=list(val_lg.keys())
    def ens(d,w):
        o=None
        for i,n in enumerate(names): o=w[i]*d[n] if o is None else o+w[i]*d[n]
        return o/(np.sum(w)+1e-12)
    if len(names)>1:
        def opt_w(d,y,k):
            best,bw=1.0,np.ones(k)/k
            for _ in range(40):
                r=minimize(lambda rw:-f1_score(y,ens(d,np.abs(rw)+1e-9).argmax(-1).numpy(),average="macro",zero_division=0),
                           np.random.dirichlet(np.ones(k)),method="Nelder-Mead",options={"maxiter":2000,"xatol":1e-5})
                if r.fun<best: best,bw=r.fun,np.abs(r.x)+1e-9
            return bw/bw.sum()
        W=opt_w(val_lg,y_val,len(names))
    else:
        W=np.ones(1)
    ens_t=ens(test_lg,W); pr=F.softmax(ens_t,-1).numpy(); yp=ens_t.argmax(-1).numpy()
    m=metrics(y_test,yp,pr,nc)
    inv={i:c for c,i in enc.items()}
    pcf=f1_score(y_test,yp,average=None,labels=list(range(nc)),zero_division=0)
    m["per_class_f1"]={inv[i]:round(float(pcf[i]),4) for i in range(nc)}
    m["best_single_macro_f1"]=round(float(max(singles)),4); m["n_test"]=int(len(y_test)); m["seeds"]=list(seeds)
    m["ensemble_weights"]={n:float(w) for n,w in zip(names,W)}
    json.dump(m,open(f"{save_root}/metrics.json","w"),indent=2)
    return m
print("run_leg ok")

## Leg 1 — Benchmark (random 20% held-out test)
Directly comparable to the main CE+FGM ensemble from NB05/06 (same splits).

In [ ]:
# ── LEG 1: Benchmark ───────────────────────────────────────────────────────
tr=pd.read_csv(f"{SPLIT_DIR}/random_train.csv"); va=pd.read_csv(f"{SPLIT_DIR}/random_val.csv"); te=pd.read_csv(f"{SPLIT_DIR}/random_test.csv")
if DEBUG:
    tr=pd.concat([g.sample(min(len(g),DEBUG_SAMPLES//5),random_state=42) for _,g in tr.groupby("label5")]).reset_index(drop=True)
    va=va.sample(min(len(va),300),random_state=42); te=te.sample(min(len(te),300),random_state=42)
classes=sorted(tr["label5"].unique()); enc={c:i for i,c in enumerate(classes)}; nc=len(classes)
print(f"LEG 1 BENCHMARK | train={len(tr):,} val={len(va):,} test={len(te):,} | classes={nc}")
bench=run_leg(tr,va,te,enc,nc,"benchmark",SEEDS,"text_clean","label5")
print("\n── Benchmark (BanglishBERT full-stack, seed-ensemble) on 20% test ──")
print(f"  macroF1={bench['macro_f1']}  wF1={bench['weighted_f1']}  acc={bench['accuracy']}  mcc={bench['mcc']}  AUROC={bench['macro_auroc']}")
print(f"  best single-seed macroF1={bench['best_single_macro_f1']}")
print("  per-class F1:", bench["per_class_f1"])

## Leg 2 — Robustness (source + script held-out)
Each config is trained from scratch on its own train+val and evaluated on its unseen test. Hard `uid`
leakage guard. No logits are reused across configs or from any other notebook.

In [ ]:
# ── LEG 2: Robustness ──────────────────────────────────────────────────────
configs=sorted({os.path.basename(p).rsplit("_",1)[0]
                for p in glob.glob(f"{SPLIT_DIR}/source_holdout_*_train.csv")+glob.glob(f"{SPLIT_DIR}/script_holdout_*_train.csv")})
print("held-out configs:",configs)
robust=[]
for tag in configs:
    rtr=pd.read_csv(f"{SPLIT_DIR}/{tag}_train.csv"); rva=pd.read_csv(f"{SPLIT_DIR}/{tag}_val.csv"); rte=pd.read_csv(f"{SPLIT_DIR}/{tag}_test.csv")
    seen=set(rtr["uid"])|set(rva["uid"]); leak=len(seen&set(rte["uid"])); assert leak==0, f"LEAK in {tag}: {leak}"
    cls=sorted(pd.concat([rtr["label5"],rva["label5"],rte["label5"]]).unique()); renc={c:i for i,c in enumerate(cls)}; rnc=len(cls)
    if DEBUG:
        rtr=pd.concat([g.sample(min(len(g),DEBUG_SAMPLES//rnc),random_state=42) for _,g in rtr.groupby("label5")]).reset_index(drop=True)
        rva=rva.sample(min(len(rva),300),random_state=42); rte=rte.sample(min(len(rte),300),random_state=42)
    print(f"\n=== {tag} | classes={rnc} train={len(rtr):,} val={len(rva):,} test={len(rte):,} | leak=0 ✅ ===")
    m=run_leg(rtr,rva,rte,renc,rnc,f"robust_{tag}",ROBUSTNESS_SEEDS,"text_clean","label5")
    m["config"]=tag; m["held_out"]=tag.replace("source_holdout_","").replace("script_holdout_","")
    robust.append(m)
    print(f"  → unseen {m['held_out']}: macroF1={m['macro_f1']} wF1={m['weighted_f1']} acc={m['accuracy']} mcc={m['mcc']}")
if robust:
    rdf=pd.DataFrame([{"config":r["config"],"held_out":r["held_out"],"n_test":r["n_test"],
                       "macro_f1":r["macro_f1"],"weighted_f1":r["weighted_f1"],"accuracy":r["accuracy"],
                       "mcc":r["mcc"],"macro_auroc":r["macro_auroc"]} for r in robust])
    rdf.to_csv(f"{OUT}/robustness_summary.csv",index=False)
    print("\n",rdf.to_string(index=False))
    print(f"\nMean across held-outs: macroF1={rdf['macro_f1'].mean():.4f}  wF1={rdf['weighted_f1'].mean():.4f}")
    print(f"✅ saved {OUT}/robustness_summary.csv")

## Leg 3 — Base-paper comparison (facebook-44k, native 5 classes, 70/15/15)
Identical facebook preprocessing/dedup/split to NB09, so the alternate method is compared to the base
paper on the same protocol.

In [ ]:
# ── LEG 3: Base-paper comparison ───────────────────────────────────────────
LABEL="label5b"
df=pd.read_csv(RAW); fb=df[df["source"]=="facebook_44001"].reset_index(drop=True).copy()
print(f"facebook rows: {len(fb):,}")
URL=re.compile(r"https?://\S+|www\.\S+"); MEN=re.compile(r"@[\w]+"); HASH=re.compile(r"#(\S+)")
ZW=re.compile(r"[\u200b\u200c\u200d\ufeff]"); WS=re.compile(r"\s+"); ELONG=re.compile(r"(.)\1{2,}")
EMO=re.compile("["+"\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
               "\U0001F1E0-\U0001F1FF\U00002700-\U000027BF\U0001F900-\U0001F9FF\U00002600-\U000026FF]+",flags=re.UNICODE)
def clean(t):
    if not isinstance(t,str): return ""
    t=unicodedata.normalize("NFKC",t); t=ZW.sub("",t); t=URL.sub(" [URL] ",t); t=MEN.sub(" [USER] ",t)
    t=HASH.sub(r" [HASHTAG] \1 ",t); t=EMO.sub(" [EMOJI] ",t); t=ELONG.sub(r"\1\1",t); return WS.sub(" ",t).strip()
fb["text_clean"]=fb["text"].apply(clean); fb=fb[fb["text_clean"].str.len()>0]
n0=len(fb); fb=fb.drop_duplicates("text_clean").reset_index(drop=True)
print(f"within-facebook dedup: {n0:,} → {len(fb):,}")
NAME={"not bully":"Not Bully","sexual":"Sexual","troll":"Troll","religious":"Religious","threat":"Threat"}
fb[LABEL]=fb["label_type"].astype(str).str.strip().str.lower().map(NAME)
fb=fb[fb[LABEL].notna()].reset_index(drop=True); assert fb[LABEL].nunique()==5
trf,tmp=train_test_split(fb,test_size=0.30,random_state=SPLIT_SEED,stratify=fb[LABEL])
vaf,tef=train_test_split(tmp,test_size=0.50,random_state=SPLIT_SEED,stratify=tmp[LABEL])
if DEBUG:
    trf=pd.concat([g.sample(min(len(g),DEBUG_SAMPLES//5),random_state=42) for _,g in trf.groupby(LABEL)]).reset_index(drop=True)
    vaf=vaf.sample(min(len(vaf),300),random_state=42); tef=tef.sample(min(len(tef),300),random_state=42)
classes_fb=sorted(fb[LABEL].unique()); enc_fb={c:i for i,c in enumerate(classes_fb)}
print(f"split 70/15/15 train={len(trf):,} val={len(vaf):,} test={len(tef):,} | enc={enc_fb}")
bp=run_leg(trf,vaf,tef,enc_fb,5,"basepaper",SEEDS,"text_clean","label5b")
print("\n"+"="*60); print(f"OURS-ALT (facebook 15% test, n={bp['n_test']:,}) vs BASE PAPER"); print("="*60)
print(f"  Macro-F1: {bp['macro_f1']:.4f}  |  base: {BASE_PAPER['macro_f1']:.4f}  Δ={bp['macro_f1']-BASE_PAPER['macro_f1']:+.4f}")
print(f"  Weighted-F1 {bp['weighted_f1']} | Acc {bp['accuracy']} | MCC {bp['mcc']} | AUROC {bp['macro_auroc']}")
print("\n  Per-class F1 (ours-alt vs base):")
for c in classes_fb:
    b=BASE_PAPER["per_class"].get(c,float("nan")); print(f"    {c:12s} {bp['per_class_f1'][c]:.4f}  vs  {b:.4f}")
json.dump({"ours_alt":bp,"base_paper":BASE_PAPER},open(f"{OUT}/basepaper_comparison.json","w"),indent=2)
print(f"✅ saved {OUT}/basepaper_comparison.json")

In [ ]:
# ── Master summary (alternate method, all three legs) ──────────────────────
print("="*64); print("ALTERNATE METHOD (BanglishBERT full-stack) — SUMMARY"); print("="*64)
print(f"  Benchmark (20% test)   macroF1={bench['macro_f1']}  wF1={bench['weighted_f1']}  acc={bench['accuracy']}")
if robust:
    print(f"  Robustness (mean)      macroF1={pd.DataFrame(robust)['macro_f1'].mean():.4f}")
    for r in robust: print(f"     - {r['held_out']:18s} macroF1={r['macro_f1']}")
print(f"  Base-paper (fb 15%)    macroF1={bp['macro_f1']}  (base {BASE_PAPER['macro_f1']}, Δ={bp['macro_f1']-BASE_PAPER['macro_f1']:+.4f})")
json.dump({"benchmark":bench,"robustness":robust,"basepaper":bp},open(f"{OUT}/altmethod_summary.json","w"),indent=2,default=float)
print(f"\n✅ all outputs under {OUT}/  (benchmark/ robust_*/ basepaper/ + *.csv/*.json)")

---
**Outputs** under `outputs/altmethod/`: `benchmark/metrics.json`, `robust_<config>/metrics.json` +
`robustness_summary.csv`, `basepaper/metrics.json` + `basepaper_comparison.json`, `altmethod_summary.json`.
NB10 reads these to add the alternate method into the paper tables.